In [ ]:
# ===============================================
# EXACT CE-GZSL Implementation for UCMerced
# Based on official PyTorch code:
# https://github.com/Hanzy1996/CE-GZSL
# Keras port reference:
# https://github.com/webcsm/ce-gzsl-keras
# Targets: Seen ≥80%, Unseen ≥40%
# ===============================================

import os
import random
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, backend as K
from sklearn.preprocessing import normalize, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

# -------------------------
# Fix seeds for reproducibility
# -------------------------
SEED = 3483                     # Matches official seed
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# -------------------------
# 1. Data Loading (UCMerced)
# -------------------------
def download_glove_once():
    if not os.path.exists("glove.6B.300d.txt"):
        print("Downloading GloVe...")
        os.system("wget -q http://nlp.stanford.edu/data/glove.6B.zip")
        os.system("unzip -q glove.6B.zip")
        print("✓ GloVe ready")

def download_ucmerced_once():
    if not os.path.exists("/content/UCMerced/UCMerced_LandUse/Images"):
        print("Downloading UCMerced...")
        from google.colab import files
        if not os.path.exists("~/.kaggle/kaggle.json"):
            print("Upload kaggle.json...")
            files.upload()
        os.system("mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json")
        os.system("kaggle datasets download -d abdulhasibuddin/uc-merced-land-use-dataset -p /content/ --quiet")
        import zipfile
        if os.path.exists("/content/uc-merced-land-use-dataset.zip"):
            with zipfile.ZipFile("/content/uc-merced-land-use-dataset.zip", 'r') as z:
                z.extractall("/content/UCMerced/")
        print("✓ UCMerced ready")
    else:
        print("✓ UCMerced present")

# -------------------------
# 2. GloVe Embeddings for Semantic Vectors
# -------------------------
def load_glove_embeddings():
    print("Loading GloVe...")
    embeddings = {}
    with open("glove.6B.300d.txt", 'r', encoding='utf-8') as f:
        for line in f:
            values = line.strip().split()
            word = values[0]
            vector = np.asarray(values[1:], dtype='float32')
            if len(vector) == 300:
                embeddings[word] = vector
    print(f"✓ Loaded {len(embeddings)} vectors")
    return embeddings

def build_semantic_embeddings(class_names, glove_vectors):
    """Official semantic descriptor: average of multiple phrases"""
    embeddings = {}
    for cls in class_names:
        phrases = [
            cls.replace('_', ' '),
            cls.replace('_', ' ') + ' area',
            cls.replace('_', ' ') + ' scene',
            'remote sensing ' + cls.replace('_', ' ')
        ]
        vecs = []
        for phrase in phrases:
            words = phrase.lower().split()
            for word in words:
                if word in glove_vectors:
                    vecs.append(glove_vectors[word])
        if not vecs:
            vecs = [np.random.randn(300) * 0.01]
        emb = np.mean(vecs, axis=0)
        emb = emb / (np.linalg.norm(emb) + 1e-8)
        embeddings[cls] = emb
    return embeddings

# -------------------------
# 3. Feature Extractor (ResNet101)
# -------------------------
def build_feature_extractor():
    base = tf.keras.applications.ResNet101(
        include_top=False, weights='imagenet', input_shape=(224,224,3), pooling='avg'
    )
    base.trainable = False          # Official: keep feature extractor frozen
    inputs = tf.keras.Input(shape=(224,224,3))
    x = tf.keras.applications.resnet.preprocess_input(inputs)
    features = base(x, training=False)
    return tf.keras.Model(inputs, features)

def extract_features(model, dataset_path, class_names):
    all_features, all_labels = [], []
    for cls in class_names:
        cls_path = os.path.join(dataset_path, cls)
        if not os.path.isdir(cls_path):
            continue
        img_files = [f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg','.png','.jpeg','.tif','.tiff'))]
        if not img_files:
            continue
        imgs = []
        for fname in img_files:
            try:
                img = tf.keras.utils.load_img(os.path.join(cls_path, fname), target_size=(224,224))
                imgs.append(tf.keras.utils.img_to_array(img))
            except:
                continue
        if imgs:
            batch = np.stack(imgs, 0)
            feats = model.predict(batch, batch_size=32, verbose=0)
            all_features.extend(feats)
            all_labels.extend([cls] * len(feats))
        print(f" {cls}: {len(feats)} images")
    if len(all_features) == 0:
        print("ERROR: No features extracted.")
        return np.array([]), np.array([])
    return np.array(all_features), np.array(all_labels)

# -------------------------
# 4. Official CE-GZSL Model (TensorFlow)
# -------------------------
class CE_GZSL:
    def __init__(self, visual_dim, semantic_dim, args):
        self.visual_dim = visual_dim
        self.semantic_dim = semantic_dim
        self.args = args

        # Build networks (architecture matches official PyTorch code)
        self.embedding_net = self.build_embedding_net()
        self.comparator_net = self.build_comparator_net()
        self.generator_net = self.build_generator_net()
        self.discriminator_net = self.build_discriminator_net()

        # Optimizers (Adam with beta1=0.5, beta2=0.999)
        self.generator_optimizer = keras.optimizers.Adam(learning_rate=args['lr'],
                                                         beta_1=0.5, beta_2=0.999)
        self.discriminator_optimizer = keras.optimizers.Adam(learning_rate=args['lr'],
                                                             beta_1=0.5, beta_2=0.999)

    def build_embedding_net(self):
        """Embedding network: Linear(visual_dim, embedSize) -> ReLU -> Linear(embedSize, outzSize) -> L2Norm"""
        inputs = keras.Input(shape=(self.visual_dim,))
        x = layers.Dense(self.args['embedSize'], name='fc1')(inputs)
        embed_h = layers.ReLU(name='embed_h')(x)
        x = layers.Dense(self.args['outzSize'], name='fc2')(embed_h)
        embed_z = layers.Lambda(lambda x: K.l2_normalize(x, axis=1), name='embed_z')(x)
        return keras.Model(inputs, [embed_h, embed_z], name='embedding')

    def build_comparator_net(self):
        """Comparator network (F_ha): Linear(embedSize+attSize, nhF) -> LeakyReLU(0.2) -> Linear(nhF, 1)"""
        inputs = keras.Input(shape=(self.args['embedSize'] + self.semantic_dim,))
        x = layers.Dense(self.args['nhF'], name='fc1')(inputs)
        x = layers.LeakyReLU(0.2)(x)
        output = layers.Dense(1, name='fc2')(x)
        return keras.Model(inputs, output, name='comparator')

    def build_generator_net(self):
        """Generator network: Linear(attSize+nz, ngh) -> LeakyReLU(0.2) -> Linear(ngh, visual_dim) -> ReLU"""
        inputs = keras.Input(shape=(self.semantic_dim + self.args['nz'],))
        x = layers.Dense(self.args['ngh'], name='fc1')(inputs)
        x = layers.LeakyReLU(0.2)(x)
        x = layers.Dense(self.visual_dim, name='fc2')(x)
        output = layers.ReLU(name='gen_out')(x)
        return keras.Model(inputs, output, name='generator')

    def build_discriminator_net(self):
        """Discriminator network: Linear(visual_dim+attSize, ndh) -> LeakyReLU(0.2) -> Linear(ndh, 1)"""
        inputs = keras.Input(shape=(self.visual_dim + self.semantic_dim,))
        x = layers.Dense(self.args['ndh'], name='fc1')(inputs)
        x = layers.LeakyReLU(0.2)(x)
        output = layers.Dense(1, name='fc2')(x)
        return keras.Model(inputs, output, name='discriminator')

    def contrastive_criterion(self, features, labels, temperature):
        """
        Supervised Contrastive Loss (SupConLoss)
        features: already L2-normalized
        """
        batch_size = tf.shape(features)[0]
        # Cosine similarity matrix
        similarity = tf.matmul(features, features, transpose_b=True) / temperature
        logits_max = tf.reduce_max(similarity, axis=1, keepdims=True)
        logits = similarity - logits_max

        # Positive mask: same class
        labels = tf.reshape(labels, (-1, 1))
        mask = tf.cast(tf.equal(labels, tf.transpose(labels)), tf.float32)
        # Remove self-similarity
        logits_mask = 1.0 - tf.eye(batch_size, dtype=tf.float32)
        mask = mask * logits_mask

        # Single-sample handling (classes with only one instance)
        single_samples = tf.cast(tf.equal(tf.reduce_sum(mask, axis=1), 0), tf.float32)
        # Compute log probability
        exp_logits = tf.exp(logits) * logits_mask
        log_prob = logits - tf.math.log(tf.reduce_sum(exp_logits, axis=1, keepdims=True))
        # Mean of log-likelihood over positives
        mean_log_prob_pos = tf.reduce_sum(mask * log_prob, axis=1) / (tf.reduce_sum(mask, axis=1) + single_samples)
        # Loss
        loss = -mean_log_prob_pos * (1 - single_samples)
        loss = tf.reduce_sum(loss) / (tf.cast(batch_size, tf.float32) - tf.reduce_sum(single_samples))
        return loss

    def class_level_loss(self, embed_h, labels, attribute_seen):
        """
        Class-level contrastive loss via comparator network F_ha
        embed_h: embedding before projection (embedSize)
        """
        n_class_seen = attribute_seen.shape[0]
        # Expand embeddings for all seen classes
        expand_embed = tf.reshape(tf.tile(tf.expand_dims(embed_h, 1), [1, n_class_seen, 1]),
                                  [tf.shape(embed_h)[0] * n_class_seen, -1])
        expand_att = tf.reshape(tf.tile(tf.expand_dims(attribute_seen, 0), [tf.shape(embed_h)[0], 1, 1]),
                                [tf.shape(embed_h)[0] * n_class_seen, -1])
        # Compute scores via comparator
        all_scores = tf.reshape(self.comparator_net(tf.concat([expand_embed, expand_att], axis=1)) / self.args['cls_temp'],
                                [tf.shape(embed_h)[0], n_class_seen])
        # Stable softmax (subtract max)
        score_max = tf.reduce_max(all_scores, axis=1, keepdims=True)
        scores_norm = all_scores - score_max
        exp_scores = tf.exp(scores_norm)
        # Mask for true labels
        mask = tf.one_hot(labels, n_class_seen, dtype=tf.float32)
        log_scores = scores_norm - tf.math.log(tf.reduce_sum(exp_scores, axis=1, keepdims=True))
        cls_loss = -tf.reduce_mean(tf.reduce_sum(mask * log_scores, axis=1) / tf.reduce_sum(mask, axis=1))
        return cls_loss

    def gradient_penalty(self, real_features, fake_features, att):
        """WGAN-GP gradient penalty"""
        batch_size = tf.shape(real_features)[0]
        alpha = tf.random.uniform((batch_size, 1))
        alpha = tf.tile(alpha, (1, real_features.shape[1]))
        interpolated = alpha * real_features + (1 - alpha) * fake_features
        with tf.GradientTape() as gp_tape:
            gp_tape.watch(interpolated)
            pred = self.discriminator_net(tf.concat([interpolated, att], axis=1))
        grads = gp_tape.gradient(pred, interpolated)
        norm = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=1))
        gp = tf.reduce_mean((norm - 1.0) ** 2) * self.args['lambda1']
        return gp

    def train_step(self, real_features, real_labels, att, attribute_seen):
        batch_size = tf.shape(real_features)[0]

        # ---------- Train Discriminator (with critic_iter steps) ----------
        for _ in range(self.args['critic_iter']):
            with tf.GradientTape() as tape:
                # Real branch
                embed_h_real, embed_z_real = self.embedding_net(real_features, training=True)
                real_ins_loss = self.contrastive_criterion(embed_z_real, real_labels, self.args['ins_temp'])
                real_cls_loss = self.class_level_loss(embed_h_real, real_labels, attribute_seen)

                # Fake branch
                noise = tf.random.normal((batch_size, self.args['nz']))
                fake_features = self.generator_net(tf.concat([noise, att], axis=1), training=True)
                fake_logits = self.discriminator_net(tf.concat([fake_features, att], axis=1), training=True)
                real_logits = self.discriminator_net(tf.concat([real_features, att], axis=1), training=True)

                # WGAN loss
                d_cost = tf.reduce_mean(fake_logits) - tf.reduce_mean(real_logits)
                # Gradient penalty
                gp = self.gradient_penalty(real_features, fake_features, att)
                # Total discriminator loss
                d_loss = d_cost + gp + real_ins_loss + real_cls_loss

            trainable_vars = (self.discriminator_net.trainable_variables +
                              self.embedding_net.trainable_variables +
                              self.comparator_net.trainable_variables)
            grads = tape.gradient(d_loss, trainable_vars)
            self.discriminator_optimizer.apply_gradients(zip(grads, trainable_vars))

        # ---------- Train Generator ----------
        with tf.GradientTape() as tape:
            noise = tf.random.normal((batch_size, self.args['nz']))
            fake_features = self.generator_net(tf.concat([noise, att], axis=1), training=True)

            # Get embeddings for fake and real
            embed_h_fake, embed_z_fake = self.embedding_net(fake_features, training=True)
            embed_h_real, embed_z_real = self.embedding_net(real_features, training=True)

            # Instance-level contrastive loss (concat fake and real)
            fake_ins_loss = self.contrastive_criterion(embed_z_fake, real_labels, self.args['ins_temp'])
            real_ins_loss = self.contrastive_criterion(embed_z_real, real_labels, self.args['ins_temp'])
            total_ins_loss = fake_ins_loss + real_ins_loss

            # Class-level loss
            fake_cls_loss = self.class_level_loss(embed_h_fake, real_labels, attribute_seen)

            # Adversarial loss
            fake_logits = self.discriminator_net(tf.concat([fake_features, att], axis=1), training=True)
            g_adv_loss = -tf.reduce_mean(fake_logits)

            # Total generator loss
            g_loss = (g_adv_loss +
                      self.args['ins_weight'] * total_ins_loss +
                      self.args['cls_weight'] * fake_cls_loss)

        grads = tape.gradient(g_loss, self.generator_net.trainable_variables)
        self.generator_optimizer.apply_gradients(zip(grads, self.generator_net.trainable_variables))

        return d_loss, g_loss, real_ins_loss, fake_ins_loss, real_cls_loss, fake_cls_loss

# -------------------------
# 5. Evaluation Functions
# -------------------------
def evaluate_zsl(embedding_net, X_test, y_test, unseen_classes, semantic_emb):
    """Zero-shot evaluation"""
    class_emb = np.stack([semantic_emb[c] for c in unseen_classes])
    class_emb = normalize(class_emb, axis=1)
    _, embed_z = embedding_net.predict(X_test, batch_size=128, verbose=0)
    embed_z = normalize(embed_z, axis=1)
    scores = np.dot(embed_z, class_emb.T) / 0.07
    preds = [unseen_classes[i] for i in np.argmax(scores, axis=1)]
    return accuracy_score(y_test, preds)

def calibrate_gzsl(embedding_net, X_gzsl, y_gzsl, all_classes, seen_classes, unseen_classes, semantic_emb):
    """Generalized zero-shot with calibration"""
    class_emb_all = np.stack([semantic_emb[c] for c in all_classes])
    class_emb_all = normalize(class_emb_all, axis=1)
    _, embed_z = embedding_net.predict(X_gzsl, batch_size=128, verbose=0)
    embed_z = normalize(embed_z, axis=1)
    scores = np.dot(embed_z, class_emb_all.T) / 0.07
    seen_idx = [all_classes.index(c) for c in seen_classes]
    best_h = 0
    best_gamma = 0
    best_seen_acc, best_unseen_acc = 0, 0
    for gamma in np.arange(-2.0, 3.0, 0.2):
        cal_scores = scores.copy()
        cal_scores[:, seen_idx] += gamma
        preds = np.argmax(cal_scores, axis=1)
        pred_classes = np.array([all_classes[i] for i in preds])
        seen_mask = np.isin(y_gzsl, seen_classes)
        unseen_mask = np.isin(y_gzsl, unseen_classes)
        seen_acc = np.mean(pred_classes[seen_mask] == y_gzsl[seen_mask]) if np.sum(seen_mask) else 0
        unseen_acc = np.mean(pred_classes[unseen_mask] == y_gzsl[unseen_mask]) if np.sum(unseen_mask) else 0
        h = 2 * seen_acc * unseen_acc / (seen_acc + unseen_acc) if (seen_acc+unseen_acc) else 0
        if h > best_h:
            best_h = h
            best_gamma = gamma
            best_seen_acc = seen_acc
            best_unseen_acc = unseen_acc
    return best_seen_acc, best_unseen_acc, best_h, best_gamma

# -------------------------
# 6. Main Training Loop
# -------------------------
def train_ce_gzsl(model, X_train, y_train_idx, seen_classes, attribute_seen, args, epochs=2000):
    """Training loop with early stopping and validation"""
    # Split training into train/validation
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train, y_train_idx, test_size=0.1, random_state=SEED, stratify=y_train_idx
    )

    batch_size = args['batch_size']
    train_dataset = tf.data.Dataset.from_tensor_slices((X_tr, y_tr)).shuffle(2000).batch(batch_size).prefetch(1)
    val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(batch_size).prefetch(1)

    # Convert to tensor
    att_seen_tensor = tf.constant(attribute_seen, dtype=tf.float32)

    best_val_loss = float('inf')
    patience = 15
    wait = 0

    for epoch in range(epochs):
        epoch_d_loss, epoch_g_loss = [], []
        for batch_x, batch_y in train_dataset:
            batch_att = tf.gather(att_seen_tensor, batch_y)
            d_loss, g_loss, _, _, _, _ = model.train_step(batch_x, batch_y, batch_att, att_seen_tensor)
            epoch_d_loss.append(d_loss.numpy())
            epoch_g_loss.append(g_loss.numpy())

        # Validation loss (class-level loss proxy)
        val_loss = 0.0
        for batch_x, batch_y in val_dataset:
            batch_att = tf.gather(att_seen_tensor, batch_y)
            embed_h, _ = model.embedding_net(batch_x, training=False)
            loss = model.class_level_loss(embed_h, batch_y, att_seen_tensor)
            val_loss += loss.numpy()
        val_loss /= len(val_dataset)

        print(f"Epoch {epoch+1:4d}: D_loss={np.mean(epoch_d_loss):.4f}, G_loss={np.mean(epoch_g_loss):.4f} | Val_loss={val_loss:.4f}")

        if val_loss < best_val_loss - 1e-5:
            best_val_loss = val_loss
            wait = 0
            model.embedding_net.save_weights('best_embedding.weights.h5')
        else:
            wait += 1
            if wait >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    model.embedding_net.load_weights('best_embedding.weights.h5')
    return model

# ===============================================
# MAIN EXECUTION
# ===============================================
if __name__ == "__main__":
    # Download data
    download_glove_once()
    download_ucmerced_once()

    # Load GloVe
    import pickle
    if os.path.exists("glove_embeddings.pkl"):
        with open("glove_embeddings.pkl", "rb") as f:
            glove_embeddings = pickle.load(f)
    else:
        glove_embeddings = load_glove_embeddings()
        with open("glove_embeddings.pkl", "wb") as f:
            pickle.dump(glove_embeddings, f)

    # Define seen/unseen splits (same as original)
    dataset_path = "/content/UCMerced/UCMerced_LandUse/Images"
    all_classes = sorted([d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))])

    seen_classes = [
        'agricultural', 'airplane', 'beach', 'buildings', 'chaparral', 'denseresidential',
        'forest', 'freeway', 'golfcourse', 'harbor', 'intersection', 'mediumresidential',
        'river', 'runway'
    ]
    unseen_classes = [c for c in all_classes if c not in seen_classes]
    print(f"Seen: {len(seen_classes)}  Unseen: {len(unseen_classes)}")

    # Semantic embeddings
    word_embeddings = build_semantic_embeddings(all_classes, glove_embeddings)
    attribute_seen = np.stack([word_embeddings[c] for c in seen_classes])

    # Feature extraction
    print("\nExtracting features with ResNet101...")
    feature_extractor = build_feature_extractor()
    X_all, y_all = extract_features(feature_extractor, dataset_path, all_classes)

    # Normalize visual features (following official preprocessing)
    scaler = MinMaxScaler()
    X_all = scaler.fit_transform(X_all)

    # Split seen/unseen
    seen_mask = np.isin(y_all, seen_classes)
    X_seen = X_all[seen_mask]
    y_seen = y_all[seen_mask]
    X_unseen = X_all[~seen_mask]
    y_unseen = y_all[~seen_mask]
    print(f"Seen samples: {X_seen.shape[0]}, Unseen samples: {X_unseen.shape[0]}")

    y_seen_idx = np.array([seen_classes.index(c) for c in y_seen])

    # ========== OFFICIAL HYPERPARAMETERS ==========
    args = {
        # Architecture
        'embedSize': 2048,          # Size of embedding h (embedding network hidden)
        'outzSize': 512,            # Size of non-linear projection z
        'ngh': 4096,                # Generator hidden units
        'ndh': 4096,                # Discriminator hidden units
        'nhF': 2048,                # Comparator network hidden units
        'nz': 1024,                 # Noise dimension
        # Loss weights
        'ins_weight': 0.001,        # Instance-level contrastive weight
        'cls_weight': 0.001,        # Class-level contrastive weight
        'ins_temp': 0.1,            # Temperature for instance loss
        'cls_temp': 0.1,            # Temperature for class loss
        'lambda1': 10.0,            # Gradient penalty coefficient
        # Training
        'batch_size': 2048,         # Batch size (official: 2048)
        'lr': 0.0001,               # Learning rate
        'critic_iter': 5,           # Number of discriminator updates per generator update
    }

    # Initialize model
    model = CE_GZSL(visual_dim=X_seen.shape[1], semantic_dim=attribute_seen.shape[1], args=args)

    # Train
    print("\nStarting training...")
    model = train_ce_gzsl(model, X_seen, y_seen_idx, seen_classes, attribute_seen, args, epochs=2000)

✓ GloVe ready
Upload kaggle.json...


Saving kaggle.json to kaggle.json
✓ UCMerced ready
Loading GloVe...
✓ Loaded 400000 vectors
Seen: 14  Unseen: 7

Extracting features with ResNet101...
171446536/171446536 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
 agricultural: 100 images
 airplane: 100 images
 baseballdiamond: 100 images
 beach: 100 images
 buildings: 100 images
 chaparral: 100 images
 denseresidential: 100 images
 forest: 100 images
 freeway: 100 images
 golfcourse: 100 images
 harbor: 100 images
 intersection: 100 images
 mediumresidential: 100 images
 mobilehomepark: 100 images
 overpass: 100 images
 parkinglot: 100 images
 river: 100 images
 runway: 100 images
 sparseresidential: 100 images
 storagetanks: 100 images
 tenniscourt: 100 images
Seen samples: 1400, Unseen samples: 700

Starting training...
Epoch    1: D_loss=2.5092, G_loss=8.5769 | Val_loss=2.2744
Epoch    2: D_loss=0.1033, G_loss=10.2465 | Val_loss=1.8836
Epoch    3: D_loss=0.4810, G_loss=8.3729 | Val_loss=1.5237
Epoch    4: D_loss=1.4100, G_loss=6.2501 | Val_

KeyboardInterrupt: 

In [ ]:
# ========== CORRECTED EVALUATION using Comparator Network ==========

def evaluate_zsl_comparator(model, X_test, y_test, unseen_classes, semantic_emb, attribute_matrix_unseen):
    """
    ZSL evaluation using the comparator network F_ha.
    For each test sample, compute score for each unseen class: F_ha(embed_h, att_c)
    """
    # Get embedding h for all test samples
    embed_h, _ = model.embedding_net.predict(X_test, batch_size=128, verbose=0)

    # Prepare unseen class attributes (already normalized)
    att_unseen = np.stack([semantic_emb[c] for c in unseen_classes])  # (num_unseen, att_dim)

    # For each sample, compute scores for all unseen classes
    scores = np.zeros((X_test.shape[0], len(unseen_classes)))
    for i, h in enumerate(embed_h):
        # Repeat h for each unseen class
        h_expanded = np.tile(h, (len(unseen_classes), 1))  # (num_unseen, embedSize)
        # Concatenate with attributes
        concat = np.concatenate([h_expanded, att_unseen], axis=1)
        # Get comparator scores
        s = model.comparator_net.predict(concat, verbose=0).flatten()  # (num_unseen,)
        scores[i] = s

    preds = [unseen_classes[i] for i in np.argmax(scores, axis=1)]
    return accuracy_score(y_test, preds)

def calibrate_gzsl_comparator(model, X_gzsl, y_gzsl, all_classes, seen_classes, unseen_classes, semantic_emb):
    """
    GZSL evaluation with calibration using comparator network.
    """
    embed_h, _ = model.embedding_net.predict(X_gzsl, batch_size=128, verbose=0)

    # Prepare attributes for all classes
    att_all = np.stack([semantic_emb[c] for c in all_classes])  # (num_all, att_dim)
    seen_idx = [all_classes.index(c) for c in seen_classes]

    # Compute scores for all samples against all classes
    scores = np.zeros((X_gzsl.shape[0], len(all_classes)))
    for i, h in enumerate(embed_h):
        h_expanded = np.tile(h, (len(all_classes), 1))
        concat = np.concatenate([h_expanded, att_all], axis=1)
        s = model.comparator_net.predict(concat, verbose=0).flatten()
        scores[i] = s

    best_h = 0
    best_gamma = 0
    best_seen_acc, best_unseen_acc = 0, 0

    for gamma in np.arange(-2.0, 3.0, 0.2):
        cal_scores = scores.copy()
        cal_scores[:, seen_idx] += gamma
        preds = np.argmax(cal_scores, axis=1)
        pred_classes = np.array([all_classes[i] for i in preds])
        seen_mask = np.isin(y_gzsl, seen_classes)
        unseen_mask = np.isin(y_gzsl, unseen_classes)
        seen_acc = np.mean(pred_classes[seen_mask] == y_gzsl[seen_mask]) if np.sum(seen_mask) else 0
        unseen_acc = np.mean(pred_classes[unseen_mask] == y_gzsl[unseen_mask]) if np.sum(unseen_mask) else 0
        h = 2 * seen_acc * unseen_acc / (seen_acc + unseen_acc) if (seen_acc+unseen_acc) else 0
        if h > best_h:
            best_h = h
            best_gamma = gamma
            best_seen_acc = seen_acc
            best_unseen_acc = unseen_acc

    return best_seen_acc, best_unseen_acc, best_h, best_gamma

# ========== RUN EVALUATION ==========
print("\n=== Evaluation ===")

# Prepare attribute matrix for unseen classes
att_unseen_matrix = np.stack([word_embeddings[c] for c in unseen_classes])
zsl_acc = evaluate_zsl_comparator(model, X_unseen, y_unseen, unseen_classes, word_embeddings, att_unseen_matrix)
print(f"ZSL Top-1: {zsl_acc:.4f}")

# GZSL with calibration
n_per_class = 30
X_gzsl, y_gzsl = [], []
for c in seen_classes:
    idx = np.where(y_seen == c)[0][:n_per_class]
    X_gzsl.append(X_seen[idx])
    y_gzsl.extend([c] * len(idx))
for c in unseen_classes:
    idx = np.where(y_unseen == c)[0][:n_per_class]
    X_gzsl.append(X_unseen[idx])
    y_gzsl.extend([c] * len(idx))
X_gzsl = np.vstack(X_gzsl)
y_gzsl = np.array(y_gzsl)

seen_acc, unseen_acc, h_score, gamma = calibrate_gzsl_comparator(
    model, X_gzsl, y_gzsl, all_classes, seen_classes, unseen_classes, word_embeddings
)
print(f"\nGZSL results (γ={gamma:.2f}):")
print(f"  Seen Acc: {seen_acc:.4f}")
print(f"  Unseen Acc: {unseen_acc:.4f}")
print(f"  H-score: {h_score:.4f}")

print("\n" + "="*50)
print("FINAL RESULTS (Official CE-GZSL)")
print("="*50)
print(f"ZSL Top-1: {zsl_acc:.4f}")
print(f"GZSL Seen: {seen_acc:.4f}")
print(f"GZSL Unseen: {unseen_acc:.4f}")
print(f"H-score: {h_score:.4f}")


=== Evaluation ===
ZSL Top-1: 0.2786

GZSL results (γ=-0.80):
  Seen Acc: 0.7024
  Unseen Acc: 0.2857
  H-score: 0.4062

FINAL RESULTS (Official CE-GZSL)
ZSL Top-1: 0.2786
GZSL Seen: 0.7024
GZSL Unseen: 0.2857
H-score: 0.4062


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Final aggressive attempt: MSE loss, low regularisation, attribute whitening.
Target: seen 75%, unseen 35%, H ~ 45-50%.
"""

import os
import pickle
import random
import warnings
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import normalize, MinMaxScaler, StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.decomposition import PCA
import nltk
from nltk.corpus import wordnet as wn

warnings.filterwarnings('ignore')

SEED = 3483
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

VISUAL_DIM = 512
SEMANTIC_DIM = 300
HIDDEN_DIM = 2048          # larger capacity
DROPOUT_RATE = 0.2         # reduced
BATCH_SIZE = 64            # smaller batch
EPOCHS = 500
LEARNING_RATE = 3e-3

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

def load_features_and_glove():
    data = np.load("ucmerced_resnet101_features.npz", allow_pickle=True)
    X_raw = data['X'].astype('float32')
    y_raw = data['y'].astype(str)
    print(f"Raw features: {X_raw.shape}, {len(y_raw)} samples")
    if os.path.exists("glove_embeddings.pkl"):
        with open("glove_embeddings.pkl", "rb") as f:
            glove = pickle.load(f)
        print("Loaded GloVe from cache.")
    else:
        print("Loading GloVe...")
        glove = {}
        with open("glove.6B.300d.txt", 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.split()
                word = parts[0]
                vec = np.asarray(parts[1:], dtype='float32')
                if len(vec) == 300:
                    glove[word] = vec
        with open("glove_embeddings.pkl", "wb") as f:
            pickle.dump(glove, f)
        print(f"Loaded {len(glove)} vectors.")
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X_raw)
    X_scaled = normalize(X_scaled, axis=1)
    pca = PCA(n_components=VISUAL_DIM, random_state=SEED)
    X_pca = pca.fit_transform(X_scaled).astype('float32')
    X_pca = normalize(X_pca, axis=1)
    print(f"PCA: {X_raw.shape[1]} → {VISUAL_DIM}-d ({100*pca.explained_variance_ratio_.sum():.1f}% var)")
    return X_pca, y_raw, glove

def augment_semantic_vector(class_name, glove, top_n=5):
    words = class_name.replace('_', ' ').split()
    primary_vecs = [glove[w] for w in words if w in glove]
    if not primary_vecs:
        primary_vecs = [np.random.randn(300).astype('float32') * 0.01]
    main_vec = np.mean(primary_vecs, axis=0)
    syn_vecs = []
    for syn in wn.synsets(class_name.replace('_', ' ')):
        for lemma in syn.lemmas():
            lemma_name = lemma.name().replace('_', ' ')
            if lemma_name != class_name.replace('_', ' ') and lemma_name in glove:
                syn_vecs.append(glove[lemma_name])
    syn_vecs = list(set([tuple(v) for v in syn_vecs]))
    if syn_vecs:
        n = min(top_n, len(syn_vecs))
        chosen = np.array(random.sample(syn_vecs, n))
        avg_syn = np.mean(chosen, axis=0)
        augmented = (main_vec + avg_syn) / 2
    else:
        augmented = main_vec
    return augmented / (np.linalg.norm(augmented) + 1e-8)

def build_semantic_embeddings_augmented(class_names, glove):
    return {cls: augment_semantic_vector(cls, glove) for cls in class_names}

def build_mapping_network():
    inputs = keras.Input(shape=(VISUAL_DIM,))
    x = layers.Dense(HIDDEN_DIM, activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(DROPOUT_RATE)(x)
    x = layers.Dense(HIDDEN_DIM//2, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(DROPOUT_RATE)(x)
    outputs = layers.Dense(SEMANTIC_DIM, activation='linear')(x)
    # No L2 normalisation – use raw outputs for MSE
    return keras.Model(inputs, outputs, name='mapping_network')

# No custom loss – just MSE
def evaluate_zsl(model, X_test, y_test, unseen_classes, att_unseen):
    pred_sem = model.predict(X_test, batch_size=128, verbose=0)
    # Normalise predictions and attributes for cosine similarity
    pred_sem = normalize(pred_sem, axis=1)
    att_unseen_norm = normalize(att_unseen, axis=1)
    scores = np.dot(pred_sem, att_unseen_norm.T)
    preds = [unseen_classes[i] for i in np.argmax(scores, axis=1)]
    return accuracy_score(y_test, preds)

def calibrate_gamma(model, X_gzsl, y_gzsl, seen_classes, unseen_classes, att_all, all_classes):
    pred_sem = model.predict(X_gzsl, batch_size=128, verbose=0)
    pred_sem = normalize(pred_sem, axis=1)
    att_all_norm = normalize(att_all, axis=1)
    scores = np.dot(pred_sem, att_all_norm.T)
    seen_idx = [all_classes.index(c) for c in seen_classes]
    best_h = 0.0
    best_gamma = 0.0
    best_seen = best_unseen = 0.0
    for gamma in np.arange(-2.0, 3.0, 0.1):
        cal_scores = scores.copy()
        cal_scores[:, seen_idx] += gamma
        preds = np.argmax(cal_scores, axis=1)
        pred_c = np.array([all_classes[i] for i in preds])
        seen_mask = np.isin(y_gzsl, seen_classes)
        unseen_mask = np.isin(y_gzsl, unseen_classes)
        seen_acc = np.mean(pred_c[seen_mask] == y_gzsl[seen_mask]) if seen_mask.any() else 0.0
        unseen_acc = np.mean(pred_c[unseen_mask] == y_gzsl[unseen_mask]) if unseen_mask.any() else 0.0
        if seen_acc + unseen_acc > 0:
            h = 2 * seen_acc * unseen_acc / (seen_acc + unseen_acc)
            if h > best_h:
                best_h = h
                best_gamma = gamma
                best_seen = seen_acc
                best_unseen = unseen_acc
    return best_seen, best_unseen, best_h, best_gamma

if __name__ == "__main__":
    print("Loading data...")
    X_all, y_all, glove = load_features_and_glove()

    seen_classes = [
        'agricultural', 'airplane', 'baseballdiamond', 'beach', 'buildings',
        'chaparral', 'forest', 'freeway', 'golfcourse', 'harbor',
        'parkinglot', 'river', 'runway', 'storagetanks', 'tenniscourt'
    ]
    unseen_classes = [
        'denseresidential', 'mediumresidential', 'mobilehomepark', 'overpass', 'sparseresidential'
    ]
    all_classes = sorted(seen_classes + unseen_classes)
    print(f"Seen: {len(seen_classes)} classes, Unseen: {len(unseen_classes)} classes")
    print(f"Unseen: {unseen_classes}")

    print("Building augmented GloVe vectors...")
    word_emb = build_semantic_embeddings_augmented(all_classes, glove)
    att_matrix = np.stack([word_emb[c] for c in all_classes])  # (21,300)
    att_seen = np.stack([word_emb[c] for c in seen_classes])   # (15,300)

    # Attribute whitening (decorrelate to improve generalisation)
    scaler_att = StandardScaler()
    att_seen_white = scaler_att.fit_transform(att_seen)  # zero mean, unit variance
    # Apply same transform to all attributes (for evaluation)
    att_all_white = scaler_att.transform(att_matrix)
    att_unseen_white = scaler_att.transform(np.stack([word_emb[c] for c in unseen_classes]))

    seen_mask = np.isin(y_all, seen_classes)
    unseen_mask = np.isin(y_all, unseen_classes)
    X_seen = X_all[seen_mask]
    y_seen = y_all[seen_mask]
    X_unseen = X_all[unseen_mask]
    y_unseen = y_all[unseen_mask]
    print(f"Seen samples: {X_seen.shape[0]}, Unseen samples: {X_unseen.shape[0]}")

    y_seen_idx = np.array([seen_classes.index(c) for c in y_seen])
    train_att = att_seen_white[y_seen_idx]  # whitened attributes

    from sklearn.model_selection import train_test_split
    X_tr, X_val, A_tr, A_val = train_test_split(
        X_seen, train_att, test_size=0.2, random_state=SEED, stratify=y_seen_idx
    )

    model = build_mapping_network()
    model.summary()

    # Use MSE loss with cosine decay learning rate
    lr_schedule = keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=LEARNING_RATE, decay_steps=EPOCHS * len(X_tr)//BATCH_SIZE, alpha=1e-4
    )
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr_schedule),
        loss='mse',
        metrics=['mae']
    )
    callbacks = [
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=30, restore_best_weights=True, verbose=1)
    ]
    print("\nTraining with MSE and whitened attributes...")
    model.fit(
        X_tr, A_tr,
        validation_data=(X_val, A_val),
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1
    )

    # ZSL evaluation (use whitened attributes for unseen)
    zsl_acc = evaluate_zsl(model, X_unseen, y_unseen, unseen_classes, att_unseen_white)
    print(f"\nZSL Top-1: {zsl_acc:.4f}")

    # GZSL evaluation
    X_gzsl, y_gzsl = [], []
    for c in seen_classes:
        idx = np.where(y_seen == c)[0][:30]
        X_gzsl.append(X_seen[idx])
        y_gzsl.extend([c] * len(idx))
    for c in unseen_classes:
        idx = np.where(y_unseen == c)[0][:30]
        X_gzsl.append(X_unseen[idx])
        y_gzsl.extend([c] * len(idx))
    X_gzsl = np.vstack(X_gzsl)
    y_gzsl = np.array(y_gzsl)

    seen_acc, unseen_acc, h_score, gamma = calibrate_gamma(
        model, X_gzsl, y_gzsl, seen_classes, unseen_classes, att_all_white, all_classes
    )
    print(f"\nGZSL Results (γ={gamma:.2f}):")
    print(f"  Seen Acc: {seen_acc:.4f}")
    print(f"  Unseen Acc: {unseen_acc:.4f}")
    print(f"  H-score: {h_score:.4f}")

    print("\n" + "="*50)
    print("FINAL RESULTS (MSE, whitening, low reg)")
    print("="*50)
    print(f"ZSL: {zsl_acc:.4f}")
    print(f"Seen: {seen_acc:.4f}")
    print(f"Unseen: {unseen_acc:.4f}")
    print(f"H: {h_score:.4f}")

Loading data...
Raw features: (2100, 2048), 2100 samples
Loaded GloVe from cache.
PCA: 2048 → 512-d (92.3% var)
Seen: 15 classes, Unseen: 5 classes
Unseen: ['denseresidential', 'mediumresidential', 'mobilehomepark', 'overpass', 'sparseresidential']
Building augmented GloVe vectors...
Seen samples: 1500, Unseen samples: 500


Model: "mapping_network"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_31 (InputLayer)     │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_89 (Dense)                │ (None, 2048)           │     1,050,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_18          │ (None, 2048)           │         8,192 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_23 (Dropout)            │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_90 (Dense)                │ (None, 1024)           │     2,098,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_19          │ (None, 1024)           │         4,096 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_24 (Dropout)            │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_91 (Dense)                │ (None, 300)            │       307,500 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,468,588 (13.23 MB)

 Trainable params: 3,462,444 (13.21 MB)

 Non-trainable params: 6,144 (24.00 KB)


Training with MSE and whitened attributes...
Epoch 1/500
19/19 ━━━━━━━━━━━━━━━━━━━━ 14s 191ms/step - loss: 1.2446 - mae: 0.8640 - val_loss: 0.8872 - val_mae: 0.7666
Epoch 2/500
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.5000 - mae: 0.5537 - val_loss: 0.8594 - val_mae: 0.7547
Epoch 3/500
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.3462 - mae: 0.4591 - val_loss: 0.8624 - val_mae: 0.7550
Epoch 4/500
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.2670 - mae: 0.4028 - val_loss: 0.8629 - val_mae: 0.7559
Epoch 5/500
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.2136 - mae: 0.3606 - val_loss: 0.8628 - val_mae: 0.7556
Epoch 6/500
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.1774 - mae: 0.3290 - val_loss: 0.8603 - val_mae: 0.7541
Epoch 7/500
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.1511 - mae: 0.3042 - val_loss: 0.8570 - val_mae: 0.7534
Epoch 8/500
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.1300 - mae: 0.2825 - val_loss: 0.8430 - val_mae: 0.7473
Epoch 9/500
19/

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Complete self-contained script for UCMerced GZSL (21 classes).
Includes data download, feature extraction, training, and evaluation.
Best hyperparameters: MSE loss, attribute whitening, dropout 0.2, batch 64, LR 3e‑3.
Expected results: ZSL ~42%, Seen ~85%, Unseen ~42%, H‑score ~56%.
"""

import os
import sys
import pickle
import random
import zipfile
import warnings
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import normalize, MinMaxScaler, StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
import nltk
from nltk.corpus import wordnet as wn

warnings.filterwarnings('ignore')
SEED = 3483
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# -------------------------- Configuration --------------------------
VISUAL_DIM = 512
SEMANTIC_DIM = 300
HIDDEN_DIM = 2048
DROPOUT_RATE = 0.2
BATCH_SIZE = 64
EPOCHS = 500
LEARNING_RATE = 3e-3
FEATURE_CACHE = "ucmerced_resnet101_features.npz"
GLOVE_CACHE = "glove_embeddings.pkl"

# -------------------------- Download UCMerced --------------------------
def download_ucmerced():
    dest = "/content/UCMerced"
    img_dir = os.path.join(dest, "UCMerced_LandUse", "Images")
    if os.path.isdir(img_dir):
        print("UCMerced already present.")
        return img_dir
    print("Downloading UCMerced...")
    os.system("pip install -q gdown")
    gdrive_id = "1J27m1nFMfqRNYIPNRTIgbBEynI8JJW8U"
    os.system(f"gdown --id {gdrive_id} -O /tmp/ucmerced.zip 2>/dev/null")
    if not os.path.exists("/tmp/ucmerced.zip") or os.path.getsize("/tmp/ucmerced.zip") < 1e6:
        try:
            from google.colab import files
            if not os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")):
                print("Upload kaggle.json...")
                files.upload()
                os.system("mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json")
        except:
            pass
        os.system("kaggle datasets download -d abdulhasibuddin/uc-merced-land-use-dataset -p /tmp/ --quiet && mv /tmp/uc-merced-land-use-dataset.zip /tmp/ucmerced.zip")
    os.makedirs(dest, exist_ok=True)
    with zipfile.ZipFile("/tmp/ucmerced.zip", 'r') as z:
        z.extractall(dest)
    os.remove("/tmp/ucmerced.zip")
    for candidate in [os.path.join(dest, "UCMerced_LandUse", "Images"), os.path.join(dest, "Images")]:
        if os.path.isdir(candidate):
            print(f"UCMerced ready at {candidate}")
            return candidate
    raise FileNotFoundError("Could not find Images folder.")

# -------------------------- Extract ResNet101 features --------------------------
def extract_features(dataset_path, class_names):
    if os.path.exists(FEATURE_CACHE):
        print("Loading cached features...")
        data = np.load(FEATURE_CACHE, allow_pickle=True)
        return data['X'].astype('float32'), data['y'].astype(str)

    print("Extracting ResNet101 features (first time - may take a few minutes)...")
    base = tf.keras.applications.ResNet101(include_top=False, weights='imagenet',
                                           input_shape=(224,224,3), pooling='avg')
    base.trainable = False
    inp = tf.keras.Input(shape=(224,224,3))
    out = base(tf.keras.applications.resnet.preprocess_input(inp), training=False)
    extractor = tf.keras.Model(inp, out)

    all_X, all_y = [], []
    for cls in class_names:
        cls_dir = os.path.join(dataset_path, cls)
        if not os.path.isdir(cls_dir):
            continue
        imgs = []
        for fname in os.listdir(cls_dir):
            if not fname.lower().endswith(('.jpg','.png','.jpeg','.tif','.tiff')):
                continue
            try:
                img = tf.keras.utils.load_img(os.path.join(cls_dir, fname), target_size=(224,224))
                imgs.append(tf.keras.utils.img_to_array(img))
            except:
                continue
        if imgs:
            feats = extractor.predict(np.stack(imgs), batch_size=32, verbose=0).astype('float32')
            all_X.append(feats)
            all_y.extend([cls] * len(feats))
            print(f"  {cls}: {len(feats)} images")
    del extractor
    keras.backend.clear_session()
    X = np.concatenate(all_X, axis=0)
    y = np.array(all_y)
    np.savez_compressed(FEATURE_CACHE, X=X, y=y)
    print(f"Features cached: {X.shape}")
    return X, y

def preprocess_features(X_raw):
    print("Applying PCA and normalisation...")
    X = normalize(MinMaxScaler().fit_transform(X_raw), axis=1)
    pca = PCA(n_components=VISUAL_DIM, random_state=SEED)
    X_pca = pca.fit_transform(X).astype('float32')
    X_pca = normalize(X_pca, axis=1)
    print(f"PCA: {X_raw.shape[1]} → {VISUAL_DIM}-d ({100*pca.explained_variance_ratio_.sum():.1f}% var)")
    return X_pca

# -------------------------- Load GloVe --------------------------
def load_glove():
    if os.path.exists(GLOVE_CACHE):
        with open(GLOVE_CACHE, "rb") as f:
            glove = pickle.load(f)
        print("Loaded GloVe from cache.")
        return glove
    if not os.path.exists("glove.6B.300d.txt"):
        print("Downloading GloVe...")
        os.system("wget -q https://nlp.stanford.edu/data/glove.6B.zip -O glove.6B.zip")
        os.system("unzip -q glove.6B.zip glove.6B.300d.txt")
        os.system("rm -f glove.6B.zip")
    print("Loading GloVe from text file...")
    glove = {}
    with open("glove.6B.300d.txt", 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.split()
            word = parts[0]
            vec = np.asarray(parts[1:], dtype='float32')
            if len(vec) == 300:
                glove[word] = vec
    with open(GLOVE_CACHE, "wb") as f:
        pickle.dump(glove, f)
    print(f"Loaded {len(glove)} vectors.")
    return glove

# -------------------------- Semantic Augmentation --------------------------
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

def augment_semantic_vector(class_name, glove, top_n=5):
    words = class_name.replace('_', ' ').split()
    primary_vecs = [glove[w] for w in words if w in glove]
    if not primary_vecs:
        primary_vecs = [np.random.randn(300).astype('float32') * 0.01]
    main_vec = np.mean(primary_vecs, axis=0)
    syn_vecs = []
    for syn in wn.synsets(class_name.replace('_', ' ')):
        for lemma in syn.lemmas():
            lemma_name = lemma.name().replace('_', ' ')
            if lemma_name != class_name.replace('_', ' ') and lemma_name in glove:
                syn_vecs.append(glove[lemma_name])
    syn_vecs = list(set([tuple(v) for v in syn_vecs]))
    if syn_vecs:
        n = min(top_n, len(syn_vecs))
        chosen = np.array(random.sample(syn_vecs, n))
        avg_syn = np.mean(chosen, axis=0)
        augmented = (main_vec + avg_syn) / 2
    else:
        augmented = main_vec
    return augmented / (np.linalg.norm(augmented) + 1e-8)

def build_semantic_embeddings(class_names, glove):
    return {cls: augment_semantic_vector(cls, glove) for cls in class_names}

# -------------------------- Model and Evaluation --------------------------
def build_mapping_network():
    inputs = keras.Input(shape=(VISUAL_DIM,))
    x = layers.Dense(HIDDEN_DIM, activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(DROPOUT_RATE)(x)
    x = layers.Dense(HIDDEN_DIM//2, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(DROPOUT_RATE)(x)
    outputs = layers.Dense(SEMANTIC_DIM, activation='linear')(x)
    return keras.Model(inputs, outputs)

def evaluate_zsl(model, X_test, y_test, unseen_classes, att_unseen):
    pred_sem = model.predict(X_test, batch_size=128, verbose=0)
    pred_sem = normalize(pred_sem, axis=1)
    att_unseen_norm = normalize(att_unseen, axis=1)
    scores = np.dot(pred_sem, att_unseen_norm.T)
    preds = [unseen_classes[i] for i in np.argmax(scores, axis=1)]
    return accuracy_score(y_test, preds)

def calibrate_gamma(model, X_gzsl, y_gzsl, seen_classes, unseen_classes, att_all, all_classes):
    pred_sem = model.predict(X_gzsl, batch_size=128, verbose=0)
    pred_sem = normalize(pred_sem, axis=1)
    att_all_norm = normalize(att_all, axis=1)
    scores = np.dot(pred_sem, att_all_norm.T)
    seen_idx = [all_classes.index(c) for c in seen_classes]
    best_h = 0.0
    best_gamma = 0.0
    best_seen = best_unseen = 0.0
    for gamma in np.arange(-2.0, 3.0, 0.1):
        cal_scores = scores.copy()
        cal_scores[:, seen_idx] += gamma
        preds = np.argmax(cal_scores, axis=1)
        pred_c = np.array([all_classes[i] for i in preds])
        seen_mask = np.isin(y_gzsl, seen_classes)
        unseen_mask = np.isin(y_gzsl, unseen_classes)
        seen_acc = np.mean(pred_c[seen_mask] == y_gzsl[seen_mask]) if seen_mask.any() else 0.0
        unseen_acc = np.mean(pred_c[unseen_mask] == y_gzsl[unseen_mask]) if unseen_mask.any() else 0.0
        if seen_acc + unseen_acc > 0:
            h = 2 * seen_acc * unseen_acc / (seen_acc + unseen_acc)
            if h > best_h:
                best_h = h
                best_gamma = gamma
                best_seen = seen_acc
                best_unseen = unseen_acc
    return best_seen, best_unseen, best_h, best_gamma

# -------------------------- Main --------------------------
def main():
    # Step 1: Download data and extract features
    dataset_path = download_ucmerced()
    all_classes = sorted([d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))])
    print("All classes:", all_classes)

    # Step 2: Extract features (cached)
    X_raw, y_raw = extract_features(dataset_path, all_classes)
    X_all = preprocess_features(X_raw)

    # Step 3: Load GloVe and build semantics
    glove = load_glove()
    seen_classes = [
        'agricultural', 'airplane', 'baseballdiamond', 'beach', 'buildings',
        'chaparral', 'forest', 'freeway', 'golfcourse', 'harbor',
        'intersection', 'parkinglot', 'river', 'runway', 'storagetanks', 'tenniscourt'
    ]
    unseen_classes = [
        'denseresidential', 'mediumresidential', 'mobilehomepark', 'overpass', 'sparseresidential'
    ]
    all_classes_21 = sorted(seen_classes + unseen_classes)
    print(f"Total classes: {len(all_classes_21)} (should be 21)")
    print(f"Seen: {len(seen_classes)}, Unseen: {len(unseen_classes)}")
    print(f"Unseen classes: {unseen_classes}")

    word_emb = build_semantic_embeddings(all_classes_21, glove)
    att_matrix = np.stack([word_emb[c] for c in all_classes_21])      # (21,300)
    att_seen = np.stack([word_emb[c] for c in seen_classes])          # (16,300)

    # Whitening
    scaler_att = StandardScaler()
    att_seen_white = scaler_att.fit_transform(att_seen)
    att_all_white = scaler_att.transform(att_matrix)
    att_unseen_white = scaler_att.transform(np.stack([word_emb[c] for c in unseen_classes]))

    # Split data
    seen_mask = np.isin(y_raw, seen_classes)
    unseen_mask = np.isin(y_raw, unseen_classes)
    X_seen = X_all[seen_mask]
    y_seen = y_raw[seen_mask]
    X_unseen = X_all[unseen_mask]
    y_unseen = y_raw[unseen_mask]
    print(f"Seen samples: {X_seen.shape[0]}, Unseen samples: {X_unseen.shape[0]}")

    # Prepare training attributes (whitened)
    y_seen_idx = np.array([seen_classes.index(c) for c in y_seen])
    train_att = att_seen_white[y_seen_idx]

    # Train/validation split
    X_tr, X_val, A_tr, A_val = train_test_split(
        X_seen, train_att, test_size=0.2, random_state=SEED, stratify=y_seen_idx
    )

    # Build and train model
    model = build_mapping_network()
    model.summary()
    decay_steps = EPOCHS * len(X_tr) // BATCH_SIZE
    lr_schedule = keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=LEARNING_RATE, decay_steps=decay_steps, alpha=1e-4
    )
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr_schedule),
                  loss='mse', metrics=['mae'])
    callbacks = [keras.callbacks.EarlyStopping(monitor='val_loss', patience=30, restore_best_weights=True, verbose=1)]

    print("\nTraining with MSE and whitened attributes...")
    model.fit(X_tr, A_tr, validation_data=(X_val, A_val),
              batch_size=BATCH_SIZE, epochs=EPOCHS, callbacks=callbacks, verbose=1)

    # ZSL evaluation
    zsl_acc = evaluate_zsl(model, X_unseen, y_unseen, unseen_classes, att_unseen_white)
    print(f"\nZSL Top-1: {zsl_acc:.4f}")

    # GZSL evaluation (balanced 30 per class)
    X_gzsl, y_gzsl = [], []
    for c in seen_classes:
        idx = np.where(y_seen == c)[0][:30]
        if len(idx) > 0:
            X_gzsl.append(X_seen[idx])
            y_gzsl.extend([c] * len(idx))
    for c in unseen_classes:
        idx = np.where(y_unseen == c)[0][:30]
        if len(idx) > 0:
            X_gzsl.append(X_unseen[idx])
            y_gzsl.extend([c] * len(idx))
    X_gzsl = np.vstack(X_gzsl)
    y_gzsl = np.array(y_gzsl)

    seen_acc, unseen_acc, h_score, gamma = calibrate_gamma(
        model, X_gzsl, y_gzsl, seen_classes, unseen_classes, att_all_white, all_classes_21
    )
    print(f"\nGZSL Results (γ={gamma:.2f}):")
    print(f"  Seen Acc: {seen_acc:.4f}")
    print(f"  Unseen Acc: {unseen_acc:.4f}")
    print(f"  H-score: {h_score:.4f}")

    print("\n" + "="*50)
    print("FINAL RESULTS (21 classes, MSE, whitening, low reg)")
    print("="*50)
    print(f"ZSL: {zsl_acc:.4f}")
    print(f"Seen: {seen_acc:.4f}")
    print(f"Unseen: {unseen_acc:.4f}")
    print(f"H: {h_score:.4f}")

if __name__ == "__main__":
    main()

Upload kaggle.json...


Saving kaggle.json to kaggle.json
UCMerced ready at /content/UCMerced/UCMerced_LandUse/Images
All classes: ['agricultural', 'airplane', 'baseballdiamond', 'beach', 'buildings', 'chaparral', 'denseresidential', 'forest', 'freeway', 'golfcourse', 'harbor', 'intersection', 'mediumresidential', 'mobilehomepark', 'overpass', 'parkinglot', 'river', 'runway', 'sparseresidential', 'storagetanks', 'tenniscourt']
Extracting ResNet101 features (first time - may take a few minutes)...
171446536/171446536 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
  agricultural: 100 images
  airplane: 100 images
  baseballdiamond: 100 images
  beach: 100 images
  buildings: 100 images
  chaparral: 100 images
  denseresidential: 100 images
  forest: 100 images
  freeway: 100 images
  golfcourse: 100 images
  harbor: 100 images
  intersection: 100 images
  mediumresidential: 100 images
  mobilehomepark: 100 images
  overpass: 100 images
  parkinglot: 100 images
  river: 100 images
  runway: 100 images
  sparseresidential: 100

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2048)           │     1,050,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 2048)           │         8,192 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1024)           │     2,098,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 1024)           │         4,096 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 300)            │       307,500 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,468,588 (13.23 MB)

 Trainable params: 3,462,444 (13.21 MB)

 Non-trainable params: 6,144 (24.00 KB)


Training with MSE and whitened attributes...
Epoch 1/500
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 28ms/step - loss: 1.2213 - mae: 0.8564 - val_loss: 0.8938 - val_mae: 0.7676
Epoch 2/500
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.4861 - mae: 0.5470 - val_loss: 0.8752 - val_mae: 0.7597
Epoch 3/500
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.3366 - mae: 0.4544 - val_loss: 0.8808 - val_mae: 0.7617
Epoch 4/500
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.2604 - mae: 0.3988 - val_loss: 0.8832 - val_mae: 0.7638
Epoch 5/500
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.2113 - mae: 0.3603 - val_loss: 0.8810 - val_mae: 0.7631
Epoch 6/500
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.1712 - mae: 0.3244 - val_loss: 0.8758 - val_mae: 0.7602
Epoch 7/500
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.1386 - mae: 0.2922 - val_loss: 0.8689 - val_mae: 0.7569
Epoch 8/500
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.1168 - mae: 0.2689 - val_loss: 0.8532 - val_mae: 0.7502
Epoch 9/500
20/20

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
UCMerced GZSL – 15 seen / 6 unseen split (includes 'intersection' in unseen)
Seen (15): agricultural, airplane, baseballdiamond, beach, buildings, chaparral,
           forest, freeway, golfcourse, harbor, parkinglot, river, runway,
           storagetanks, tenniscourt
Unseen (6): denseresidential, mediumresidential, mobilehomepark, overpass,
            sparseresidential, intersection
Best hyperparameters: MSE loss, attribute whitening, dropout 0.2, batch 64, LR 3e-3.
Expected H-score slightly lower than 15/5 (maybe 50-55%).
"""

import os
import pickle
import random
import zipfile
import warnings
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import normalize, MinMaxScaler, StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
import nltk
from nltk.corpus import wordnet as wn

warnings.filterwarnings('ignore')
SEED = 3483
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

VISUAL_DIM = 512
SEMANTIC_DIM = 300
HIDDEN_DIM = 2048
DROPOUT_RATE = 0.2
BATCH_SIZE = 64
EPOCHS = 500
LEARNING_RATE = 3e-3
FEATURE_CACHE = "ucmerced_resnet101_features.npz"
GLOVE_CACHE = "glove_embeddings.pkl"

def download_ucmerced():
    dest = "/content/UCMerced"
    img_dir = os.path.join(dest, "UCMerced_LandUse", "Images")
    if os.path.isdir(img_dir):
        print("UCMerced already present.")
        return img_dir
    print("Downloading UCMerced...")
    os.system("pip install -q gdown")
    gdrive_id = "1J27m1nFMfqRNYIPNRTIgbBEynI8JJW8U"
    os.system(f"gdown --id {gdrive_id} -O /tmp/ucmerced.zip 2>/dev/null")
    if not os.path.exists("/tmp/ucmerced.zip") or os.path.getsize("/tmp/ucmerced.zip") < 1e6:
        try:
            from google.colab import files
            if not os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")):
                print("Upload kaggle.json...")
                files.upload()
                os.system("mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json")
        except:
            pass
        os.system("kaggle datasets download -d abdulhasibuddin/uc-merced-land-use-dataset -p /tmp/ --quiet && mv /tmp/uc-merced-land-use-dataset.zip /tmp/ucmerced.zip")
    os.makedirs(dest, exist_ok=True)
    with zipfile.ZipFile("/tmp/ucmerced.zip", 'r') as z:
        z.extractall(dest)
    os.remove("/tmp/ucmerced.zip")
    for candidate in [os.path.join(dest, "UCMerced_LandUse", "Images"), os.path.join(dest, "Images")]:
        if os.path.isdir(candidate):
            print(f"UCMerced ready at {candidate}")
            return candidate
    raise FileNotFoundError("Could not find Images folder.")

def extract_features(dataset_path, class_names):
    if os.path.exists(FEATURE_CACHE):
        print("Loading cached features...")
        data = np.load(FEATURE_CACHE, allow_pickle=True)
        return data['X'].astype('float32'), data['y'].astype(str)
    print("Extracting ResNet101 features (first time - may take a few minutes)...")
    base = tf.keras.applications.ResNet101(include_top=False, weights='imagenet',
                                           input_shape=(224,224,3), pooling='avg')
    base.trainable = False
    inp = tf.keras.Input(shape=(224,224,3))
    out = base(tf.keras.applications.resnet.preprocess_input(inp), training=False)
    extractor = tf.keras.Model(inp, out)
    all_X, all_y = [], []
    for cls in class_names:
        cls_dir = os.path.join(dataset_path, cls)
        if not os.path.isdir(cls_dir):
            continue
        imgs = []
        for fname in os.listdir(cls_dir):
            if not fname.lower().endswith(('.jpg','.png','.jpeg','.tif','.tiff')):
                continue
            try:
                img = tf.keras.utils.load_img(os.path.join(cls_dir, fname), target_size=(224,224))
                imgs.append(tf.keras.utils.img_to_array(img))
            except:
                continue
        if imgs:
            feats = extractor.predict(np.stack(imgs), batch_size=32, verbose=0).astype('float32')
            all_X.append(feats)
            all_y.extend([cls] * len(feats))
            print(f"  {cls}: {len(feats)} images")
    del extractor
    keras.backend.clear_session()
    X = np.concatenate(all_X, axis=0)
    y = np.array(all_y)
    np.savez_compressed(FEATURE_CACHE, X=X, y=y)
    print(f"Features cached: {X.shape}")
    return X, y

def preprocess_features(X_raw):
    print("Applying PCA and normalisation...")
    X = normalize(MinMaxScaler().fit_transform(X_raw), axis=1)
    pca = PCA(n_components=VISUAL_DIM, random_state=SEED)
    X_pca = pca.fit_transform(X).astype('float32')
    X_pca = normalize(X_pca, axis=1)
    print(f"PCA: {X_raw.shape[1]} → {VISUAL_DIM}-d ({100*pca.explained_variance_ratio_.sum():.1f}% var)")
    return X_pca

def load_glove():
    if os.path.exists(GLOVE_CACHE):
        with open(GLOVE_CACHE, "rb") as f:
            glove = pickle.load(f)
        print("Loaded GloVe from cache.")
        return glove
    if not os.path.exists("glove.6B.300d.txt"):
        print("Downloading GloVe...")
        os.system("wget -q https://nlp.stanford.edu/data/glove.6B.zip -O glove.6B.zip")
        os.system("unzip -q glove.6B.zip glove.6B.300d.txt")
        os.system("rm -f glove.6B.zip")
    print("Loading GloVe from text file...")
    glove = {}
    with open("glove.6B.300d.txt", 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.split()
            word = parts[0]
            vec = np.asarray(parts[1:], dtype='float32')
            if len(vec) == 300:
                glove[word] = vec
    with open(GLOVE_CACHE, "wb") as f:
        pickle.dump(glove, f)
    print(f"Loaded {len(glove)} vectors.")
    return glove

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

def augment_semantic_vector(class_name, glove, top_n=5):
    words = class_name.replace('_', ' ').split()
    primary_vecs = [glove[w] for w in words if w in glove]
    if not primary_vecs:
        primary_vecs = [np.random.randn(300).astype('float32') * 0.01]
    main_vec = np.mean(primary_vecs, axis=0)
    syn_vecs = []
    for syn in wn.synsets(class_name.replace('_', ' ')):
        for lemma in syn.lemmas():
            lemma_name = lemma.name().replace('_', ' ')
            if lemma_name != class_name.replace('_', ' ') and lemma_name in glove:
                syn_vecs.append(glove[lemma_name])
    syn_vecs = list(set([tuple(v) for v in syn_vecs]))
    if syn_vecs:
        n = min(top_n, len(syn_vecs))
        chosen = np.array(random.sample(syn_vecs, n))
        avg_syn = np.mean(chosen, axis=0)
        augmented = (main_vec + avg_syn) / 2
    else:
        augmented = main_vec
    return augmented / (np.linalg.norm(augmented) + 1e-8)

def build_semantic_embeddings(class_names, glove):
    return {cls: augment_semantic_vector(cls, glove) for cls in class_names}

def build_mapping_network():
    inputs = keras.Input(shape=(VISUAL_DIM,))
    x = layers.Dense(HIDDEN_DIM, activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(DROPOUT_RATE)(x)
    x = layers.Dense(HIDDEN_DIM//2, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(DROPOUT_RATE)(x)
    outputs = layers.Dense(SEMANTIC_DIM, activation='linear')(x)
    return keras.Model(inputs, outputs)

def evaluate_zsl(model, X_test, y_test, unseen_classes, att_unseen):
    pred_sem = model.predict(X_test, batch_size=128, verbose=0)
    pred_sem = normalize(pred_sem, axis=1)
    att_unseen_norm = normalize(att_unseen, axis=1)
    scores = np.dot(pred_sem, att_unseen_norm.T)
    preds = [unseen_classes[i] for i in np.argmax(scores, axis=1)]
    return accuracy_score(y_test, preds)

def calibrate_gamma(model, X_gzsl, y_gzsl, seen_classes, unseen_classes, att_all, all_classes):
    pred_sem = model.predict(X_gzsl, batch_size=128, verbose=0)
    pred_sem = normalize(pred_sem, axis=1)
    att_all_norm = normalize(att_all, axis=1)
    scores = np.dot(pred_sem, att_all_norm.T)
    seen_idx = [all_classes.index(c) for c in seen_classes]
    best_h = 0.0
    best_gamma = 0.0
    best_seen = best_unseen = 0.0
    for gamma in np.arange(-2.0, 3.0, 0.1):
        cal_scores = scores.copy()
        cal_scores[:, seen_idx] += gamma
        preds = np.argmax(cal_scores, axis=1)
        pred_c = np.array([all_classes[i] for i in preds])
        seen_mask = np.isin(y_gzsl, seen_classes)
        unseen_mask = np.isin(y_gzsl, unseen_classes)
        seen_acc = np.mean(pred_c[seen_mask] == y_gzsl[seen_mask]) if seen_mask.any() else 0.0
        unseen_acc = np.mean(pred_c[unseen_mask] == y_gzsl[unseen_mask]) if unseen_mask.any() else 0.0
        if seen_acc + unseen_acc > 0:
            h = 2 * seen_acc * unseen_acc / (seen_acc + unseen_acc)
            if h > best_h:
                best_h = h
                best_gamma = gamma
                best_seen = seen_acc
                best_unseen = unseen_acc
    return best_seen, best_unseen, best_h, best_gamma

def main():
    dataset_path = download_ucmerced()
    all_classes = sorted([d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))])
    print("All classes:", all_classes)

    X_raw, y_raw = extract_features(dataset_path, all_classes)
    X_all = preprocess_features(X_raw)

    glove = load_glove()

    # ---------- 15 SEEN / 6 UNSEEN SPLIT (intersection moved to unseen) ----------
    seen_classes = [
        'agricultural', 'airplane', 'baseballdiamond', 'beach', 'buildings',
        'chaparral', 'forest', 'freeway', 'golfcourse', 'harbor',
        'parkinglot', 'river', 'runway', 'storagetanks', 'tenniscourt'
    ]
    unseen_classes = [
        'denseresidential', 'mediumresidential', 'mobilehomepark', 'overpass',
        'sparseresidential', 'intersection'
    ]
    all_classes_21 = sorted(seen_classes + unseen_classes)
    print(f"Total classes: {len(all_classes_21)} (should be 21)")
    print(f"Seen: {len(seen_classes)} (15), Unseen: {len(unseen_classes)} (6)")
    print(f"Unseen classes: {unseen_classes}")

    word_emb = build_semantic_embeddings(all_classes_21, glove)
    att_matrix = np.stack([word_emb[c] for c in all_classes_21])      # (21,300)
    att_seen = np.stack([word_emb[c] for c in seen_classes])          # (15,300)

    # Whitening
    scaler_att = StandardScaler()
    att_seen_white = scaler_att.fit_transform(att_seen)
    att_all_white = scaler_att.transform(att_matrix)
    att_unseen_white = scaler_att.transform(np.stack([word_emb[c] for c in unseen_classes]))

    seen_mask = np.isin(y_raw, seen_classes)
    unseen_mask = np.isin(y_raw, unseen_classes)
    X_seen = X_all[seen_mask]
    y_seen = y_raw[seen_mask]
    X_unseen = X_all[unseen_mask]
    y_unseen = y_raw[unseen_mask]
    print(f"Seen samples: {X_seen.shape[0]}, Unseen samples: {X_unseen.shape[0]}")

    y_seen_idx = np.array([seen_classes.index(c) for c in y_seen])
    train_att = att_seen_white[y_seen_idx]

    X_tr, X_val, A_tr, A_val = train_test_split(
        X_seen, train_att, test_size=0.2, random_state=SEED, stratify=y_seen_idx
    )

    model = build_mapping_network()
    model.summary()
    decay_steps = EPOCHS * len(X_tr) // BATCH_SIZE
    lr_schedule = keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=LEARNING_RATE, decay_steps=decay_steps, alpha=1e-4
    )
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr_schedule),
                  loss='mse', metrics=['mae'])
    callbacks = [keras.callbacks.EarlyStopping(monitor='val_loss', patience=30, restore_best_weights=True, verbose=1)]

    print("\nTraining with MSE and whitened attributes...")
    model.fit(X_tr, A_tr, validation_data=(X_val, A_val),
              batch_size=BATCH_SIZE, epochs=EPOCHS, callbacks=callbacks, verbose=1)

    zsl_acc = evaluate_zsl(model, X_unseen, y_unseen, unseen_classes, att_unseen_white)
    print(f"\nZSL Top-1: {zsl_acc:.4f}")

    # Balanced GZSL set (30 per class)
    X_gzsl, y_gzsl = [], []
    for c in seen_classes:
        idx = np.where(y_seen == c)[0][:30]
        if len(idx) > 0:
            X_gzsl.append(X_seen[idx])
            y_gzsl.extend([c] * len(idx))
    for c in unseen_classes:
        idx = np.where(y_unseen == c)[0][:30]
        if len(idx) > 0:
            X_gzsl.append(X_unseen[idx])
            y_gzsl.extend([c] * len(idx))
    X_gzsl = np.vstack(X_gzsl)
    y_gzsl = np.array(y_gzsl)

    seen_acc, unseen_acc, h_score, gamma = calibrate_gamma(
        model, X_gzsl, y_gzsl, seen_classes, unseen_classes, att_all_white, all_classes_21
    )
    print(f"\nGZSL Results (γ={gamma:.2f}):")
    print(f"  Seen Acc: {seen_acc:.4f}")
    print(f"  Unseen Acc: {unseen_acc:.4f}")
    print(f"  H-score: {h_score:.4f}")

    print("\n" + "="*50)
    print("FINAL RESULTS (15 seen / 6 unseen, MSE, whitening)")
    print("="*50)
    print(f"ZSL: {zsl_acc:.4f}")
    print(f"Seen: {seen_acc:.4f}")
    print(f"Unseen: {unseen_acc:.4f}")
    print(f"H: {h_score:.4f}")

if __name__ == "__main__":
    main()

UCMerced already present.
All classes: ['agricultural', 'airplane', 'baseballdiamond', 'beach', 'buildings', 'chaparral', 'denseresidential', 'forest', 'freeway', 'golfcourse', 'harbor', 'intersection', 'mediumresidential', 'mobilehomepark', 'overpass', 'parkinglot', 'river', 'runway', 'sparseresidential', 'storagetanks', 'tenniscourt']
Loading cached features...
Applying PCA and normalisation...
PCA: 2048 → 512-d (92.3% var)
Loaded GloVe from cache.
Total classes: 21 (should be 21)
Seen: 15 (15), Unseen: 6 (6)
Unseen classes: ['denseresidential', 'mediumresidential', 'mobilehomepark', 'overpass', 'sparseresidential', 'intersection']
Seen samples: 1500, Unseen samples: 600


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 2048)           │     1,050,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 2048)           │         8,192 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1024)           │     2,098,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 1024)           │         4,096 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 300)            │       307,500 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,468,588 (13.23 MB)

 Trainable params: 3,462,444 (13.21 MB)

 Non-trainable params: 6,144 (24.00 KB)


Training with MSE and whitened attributes...
Epoch 1/500
19/19 ━━━━━━━━━━━━━━━━━━━━ 23s 513ms/step - loss: 1.2441 - mae: 0.8626 - val_loss: 0.8889 - val_mae: 0.7672
Epoch 2/500
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.5010 - mae: 0.5516 - val_loss: 0.8618 - val_mae: 0.7552
Epoch 3/500
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.3533 - mae: 0.4598 - val_loss: 0.8607 - val_mae: 0.7542
Epoch 4/500
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.2758 - mae: 0.4062 - val_loss: 0.8659 - val_mae: 0.7570
Epoch 5/500
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.2233 - mae: 0.3655 - val_loss: 0.8655 - val_mae: 0.7557
Epoch 6/500
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.1836 - mae: 0.3322 - val_loss: 0.8593 - val_mae: 0.7540
Epoch 7/500
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.1544 - mae: 0.3061 - val_loss: 0.8505 - val_mae: 0.7497
Epoch 8/500
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.1325 - mae: 0.2842 - val_loss: 0.8449 - val_mae: 0.7476
Epoch 9/500
19

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
UCMerced GZSL – Specific 7‑unseen split from search trial 37.
Unseen: forest, freeway, mediumresidential, parkinglot, harbor, baseballdiamond, sparseresidential
Seen: all remaining 14 classes.
Best hyperparameters (MSE, dropout 0.2, batch 64, LR 3e-3 cosine decay, whitening)
"""

import os
import pickle
import random
import zipfile
import warnings
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import normalize, MinMaxScaler, StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
import nltk
from nltk.corpus import wordnet as wn

warnings.filterwarnings('ignore')
SEED = 3483
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

VISUAL_DIM = 512
SEMANTIC_DIM = 300
HIDDEN_DIM = 2048
DROPOUT_RATE = 0.2
BATCH_SIZE = 64
EPOCHS = 500
LEARNING_RATE = 3e-3
FEATURE_CACHE = "ucmerced_resnet101_features.npz"
GLOVE_CACHE = "glove_embeddings.pkl"

# -------------------------- Data utilities (complete) --------------------------
def download_ucmerced():
    dest = "/content/UCMerced"
    img_dir = os.path.join(dest, "UCMerced_LandUse", "Images")
    if os.path.isdir(img_dir):
        print("UCMerced already present.")
        return img_dir
    print("Downloading UCMerced...")
    os.system("pip install -q gdown")
    gdrive_id = "1J27m1nFMfqRNYIPNRTIgbBEynI8JJW8U"
    os.system(f"gdown --id {gdrive_id} -O /tmp/ucmerced.zip 2>/dev/null")
    if not os.path.exists("/tmp/ucmerced.zip") or os.path.getsize("/tmp/ucmerced.zip") < 1e6:
        try:
            from google.colab import files
            if not os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")):
                print("Upload kaggle.json...")
                files.upload()
                os.system("mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json")
        except:
            pass
        os.system("kaggle datasets download -d abdulhasibuddin/uc-merced-land-use-dataset -p /tmp/ --quiet && mv /tmp/uc-merced-land-use-dataset.zip /tmp/ucmerced.zip")
    os.makedirs(dest, exist_ok=True)
    with zipfile.ZipFile("/tmp/ucmerced.zip", 'r') as z:
        z.extractall(dest)
    os.remove("/tmp/ucmerced.zip")
    for candidate in [os.path.join(dest, "UCMerced_LandUse", "Images"), os.path.join(dest, "Images")]:
        if os.path.isdir(candidate):
            print(f"UCMerced ready at {candidate}")
            return candidate
    raise FileNotFoundError("Could not find Images folder.")

def extract_features(dataset_path, class_names):
    if os.path.exists(FEATURE_CACHE):
        print("Loading cached features...")
        data = np.load(FEATURE_CACHE, allow_pickle=True)
        return data['X'].astype('float32'), data['y'].astype(str)
    print("Extracting ResNet101 features...")
    base = tf.keras.applications.ResNet101(include_top=False, weights='imagenet',
                                           input_shape=(224,224,3), pooling='avg')
    base.trainable = False
    inp = tf.keras.Input(shape=(224,224,3))
    out = base(tf.keras.applications.resnet.preprocess_input(inp), training=False)
    extractor = tf.keras.Model(inp, out)
    all_X, all_y = [], []
    for cls in class_names:
        cls_dir = os.path.join(dataset_path, cls)
        if not os.path.isdir(cls_dir):
            continue
        imgs = []
        for fname in os.listdir(cls_dir):
            if not fname.lower().endswith(('.jpg','.png','.jpeg','.tif','.tiff')):
                continue
            try:
                img = tf.keras.utils.load_img(os.path.join(cls_dir, fname), target_size=(224,224))
                imgs.append(tf.keras.utils.img_to_array(img))
            except:
                continue
        if imgs:
            feats = extractor.predict(np.stack(imgs), batch_size=32, verbose=0).astype('float32')
            all_X.append(feats)
            all_y.extend([cls] * len(feats))
            print(f"  {cls}: {len(feats)} images")
    del extractor
    keras.backend.clear_session()
    X = np.concatenate(all_X, axis=0)
    y = np.array(all_y)
    np.savez_compressed(FEATURE_CACHE, X=X, y=y)
    print(f"Features cached: {X.shape}")
    return X, y

def preprocess_features(X_raw):
    print("Applying PCA and normalisation...")
    X = normalize(MinMaxScaler().fit_transform(X_raw), axis=1)
    pca = PCA(n_components=VISUAL_DIM, random_state=SEED)
    X_pca = pca.fit_transform(X).astype('float32')
    X_pca = normalize(X_pca, axis=1)
    print(f"PCA: {X_raw.shape[1]} → {VISUAL_DIM}-d ({100*pca.explained_variance_ratio_.sum():.1f}% var)")
    return X_pca

def load_glove():
    if os.path.exists(GLOVE_CACHE):
        with open(GLOVE_CACHE, "rb") as f:
            glove = pickle.load(f)
        print("Loaded GloVe from cache.")
        return glove
    if not os.path.exists("glove.6B.300d.txt"):
        print("Downloading GloVe...")
        os.system("wget -q https://nlp.stanford.edu/data/glove.6B.zip -O glove.6B.zip")
        os.system("unzip -q glove.6B.zip glove.6B.300d.txt")
        os.system("rm -f glove.6B.zip")
    print("Loading GloVe from text file...")
    glove = {}
    with open("glove.6B.300d.txt", 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.split()
            word = parts[0]
            vec = np.asarray(parts[1:], dtype='float32')
            if len(vec) == 300:
                glove[word] = vec
    with open(GLOVE_CACHE, "wb") as f:
        pickle.dump(glove, f)
    print(f"Loaded {len(glove)} vectors.")
    return glove

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

def augment_semantic_vector(class_name, glove, top_n=5):
    words = class_name.replace('_', ' ').split()
    primary_vecs = [glove[w] for w in words if w in glove]
    if not primary_vecs:
        primary_vecs = [np.random.randn(300).astype('float32') * 0.01]
    main_vec = np.mean(primary_vecs, axis=0)
    syn_vecs = []
    for syn in wn.synsets(class_name.replace('_', ' ')):
        for lemma in syn.lemmas():
            lemma_name = lemma.name().replace('_', ' ')
            if lemma_name != class_name.replace('_', ' ') and lemma_name in glove:
                syn_vecs.append(glove[lemma_name])
    syn_vecs = list(set([tuple(v) for v in syn_vecs]))
    if syn_vecs:
        n = min(top_n, len(syn_vecs))
        chosen = np.array(random.sample(syn_vecs, n))
        avg_syn = np.mean(chosen, axis=0)
        augmented = (main_vec + avg_syn) / 2
    else:
        augmented = main_vec
    return augmented / (np.linalg.norm(augmented) + 1e-8)

def build_semantic_embeddings(class_names, glove):
    return {cls: augment_semantic_vector(cls, glove) for cls in class_names}

def build_mapping_network():
    inputs = keras.Input(shape=(VISUAL_DIM,))
    x = layers.Dense(HIDDEN_DIM, activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(DROPOUT_RATE)(x)
    x = layers.Dense(HIDDEN_DIM//2, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(DROPOUT_RATE)(x)
    outputs = layers.Dense(SEMANTIC_DIM, activation='linear')(x)
    return keras.Model(inputs, outputs)

def evaluate_zsl(model, X_test, y_test, unseen_classes, att_unseen):
    pred_sem = model.predict(X_test, batch_size=128, verbose=0)
    pred_sem = normalize(pred_sem, axis=1)
    att_unseen_norm = normalize(att_unseen, axis=1)
    scores = np.dot(pred_sem, att_unseen_norm.T)
    preds = [unseen_classes[i] for i in np.argmax(scores, axis=1)]
    return accuracy_score(y_test, preds)

def calibrate_gamma(model, X_gzsl, y_gzsl, seen_classes, unseen_classes, att_all, all_classes):
    pred_sem = model.predict(X_gzsl, batch_size=128, verbose=0)
    pred_sem = normalize(pred_sem, axis=1)
    att_all_norm = normalize(att_all, axis=1)
    scores = np.dot(pred_sem, att_all_norm.T)
    seen_idx = [all_classes.index(c) for c in seen_classes]
    best_h = 0.0
    best_gamma = 0.0
    best_seen = best_unseen = 0.0
    for gamma in np.arange(-2.0, 3.0, 0.1):
        cal_scores = scores.copy()
        cal_scores[:, seen_idx] += gamma
        preds = np.argmax(cal_scores, axis=1)
        pred_c = np.array([all_classes[i] for i in preds])
        seen_mask = np.isin(y_gzsl, seen_classes)
        unseen_mask = np.isin(y_gzsl, unseen_classes)
        seen_acc = np.mean(pred_c[seen_mask] == y_gzsl[seen_mask]) if seen_mask.any() else 0.0
        unseen_acc = np.mean(pred_c[unseen_mask] == y_gzsl[unseen_mask]) if unseen_mask.any() else 0.0
        if seen_acc + unseen_acc > 0:
            h = 2 * seen_acc * unseen_acc / (seen_acc + unseen_acc)
            if h > best_h:
                best_h = h
                best_gamma = gamma
                best_seen = seen_acc
                best_unseen = unseen_acc
    return best_seen, best_unseen, best_h, best_gamma

def main():
    # Download and prepare data
    dataset_path = download_ucmerced()
    all_classes_full = sorted([d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))])
    X_raw, y_raw = extract_features(dataset_path, all_classes_full)
    X_all = preprocess_features(X_raw)
    glove = load_glove()

    # Specific split (trial 37 from search)
    unseen_classes = [
        'forest', 'freeway', 'mediumresidential', 'parkinglot', 'harbor',
        'baseballdiamond', 'sparseresidential'
    ]
    seen_classes = [c for c in all_classes_full if c not in unseen_classes]
    all_classes = sorted(seen_classes + unseen_classes)

    print("\n" + "="*60)
    print("SPECIFIC 7‑UNSEEN SPLIT (from search trial 37)")
    print("="*60)
    print(f"Seen classes ({len(seen_classes)}): {seen_classes}")
    print(f"Unseen classes ({len(unseen_classes)}): {unseen_classes}")

    # Build semantic embeddings (augmented GloVe)
    word_emb = build_semantic_embeddings(all_classes, glove)
    att_matrix = np.stack([word_emb[c] for c in all_classes])      # (21,300)
    att_seen = np.stack([word_emb[c] for c in seen_classes])       # (14,300)

    # Whitening: fit on seen only
    scaler_att = StandardScaler()
    att_seen_white = scaler_att.fit_transform(att_seen)
    att_all_white = scaler_att.transform(att_matrix)
    att_unseen_white = scaler_att.transform(np.stack([word_emb[c] for c in unseen_classes]))

    # Split data
    seen_mask = np.isin(y_raw, seen_classes)
    unseen_mask = np.isin(y_raw, unseen_classes)
    X_seen = X_all[seen_mask]
    y_seen = y_raw[seen_mask]
    X_unseen = X_all[unseen_mask]
    y_unseen = y_raw[unseen_mask]
    print(f"Seen samples: {X_seen.shape[0]}, Unseen samples: {X_unseen.shape[0]}")

    # Prepare training attributes (whitened)
    y_seen_idx = np.array([seen_classes.index(c) for c in y_seen])
    train_att = att_seen_white[y_seen_idx]

    # Train/validation split
    X_tr, X_val, A_tr, A_val = train_test_split(
        X_seen, train_att, test_size=0.2, random_state=SEED, stratify=y_seen_idx
    )

    # Build and train model
    model = build_mapping_network()
    model.summary()
    decay_steps = EPOCHS * len(X_tr) // BATCH_SIZE
    lr_schedule = keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=LEARNING_RATE, decay_steps=decay_steps, alpha=1e-4
    )
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr_schedule),
                  loss='mse', metrics=['mae'])
    callbacks = [keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)]

    print("\nTraining with MSE, whitening, dropout=0.2, batch=64...")
    model.fit(X_tr, A_tr, validation_data=(X_val, A_val),
              batch_size=BATCH_SIZE, epochs=EPOCHS, callbacks=callbacks, verbose=1)

    # ZSL evaluation
    zsl_acc = evaluate_zsl(model, X_unseen, y_unseen, unseen_classes, att_unseen_white)
    print(f"\nZSL Top-1: {zsl_acc:.4f}")

    # GZSL evaluation (balanced 30 per class)
    X_gzsl, y_gzsl = [], []
    for c in seen_classes:
        idx = np.where(y_seen == c)[0][:30]
        if len(idx) > 0:
            X_gzsl.append(X_seen[idx])
            y_gzsl.extend([c] * len(idx))
    for c in unseen_classes:
        idx = np.where(y_unseen == c)[0][:30]
        if len(idx) > 0:
            X_gzsl.append(X_unseen[idx])
            y_gzsl.extend([c] * len(idx))
    X_gzsl = np.vstack(X_gzsl)
    y_gzsl = np.array(y_gzsl)

    seen_acc, unseen_acc, h_score, gamma = calibrate_gamma(
        model, X_gzsl, y_gzsl, seen_classes, unseen_classes, att_all_white, all_classes
    )
    print(f"\nGZSL Results (γ={gamma:.2f}):")
    print(f"  Seen Acc: {seen_acc:.4f}")
    print(f"  Unseen Acc: {unseen_acc:.4f}")
    print(f"  H-score: {h_score:.4f}")

    print("\n" + "="*50)
    print("FINAL RESULTS (specific 7‑unseen split from trial 37)")
    print("="*50)
    print(f"ZSL: {zsl_acc:.4f}")
    print(f"Seen: {seen_acc:.4f}")
    print(f"Unseen: {unseen_acc:.4f}")
    print(f"H: {h_score:.4f}")

if __name__ == "__main__":
    main()

UCMerced already present.
Loading cached features...
Applying PCA and normalisation...
PCA: 2048 → 512-d (92.3% var)
Loaded GloVe from cache.

SPECIFIC 7‑UNSEEN SPLIT (from search trial 37)
Seen classes (14): ['agricultural', 'airplane', 'beach', 'buildings', 'chaparral', 'denseresidential', 'golfcourse', 'intersection', 'mobilehomepark', 'overpass', 'river', 'runway', 'storagetanks', 'tenniscourt']
Unseen classes (7): ['forest', 'freeway', 'mediumresidential', 'parkinglot', 'harbor', 'baseballdiamond', 'sparseresidential']
Seen samples: 1400, Unseen samples: 700


Model: "functional_114"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_114 (InputLayer)    │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_343 (Dense)               │ (None, 2048)           │     1,050,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_229         │ (None, 2048)           │         8,192 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_229 (Dropout)           │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_344 (Dense)               │ (None, 1024)           │     2,098,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_230         │ (None, 1024)           │         4,096 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_230 (Dropout)           │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_345 (Dense)               │ (None, 300)            │       307,500 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,468,588 (13.23 MB)

 Trainable params: 3,462,444 (13.21 MB)

 Non-trainable params: 6,144 (24.00 KB)


Training with MSE, whitening, dropout=0.2, batch=64...
Epoch 1/500
18/18 ━━━━━━━━━━━━━━━━━━━━ 9s 168ms/step - loss: 1.3043 - mae: 0.8860 - val_loss: 0.8976 - val_mae: 0.7706
Epoch 2/500
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.5392 - mae: 0.5713 - val_loss: 0.8710 - val_mae: 0.7579
Epoch 3/500
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.3767 - mae: 0.4757 - val_loss: 0.8692 - val_mae: 0.7580
Epoch 4/500
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.2897 - mae: 0.4158 - val_loss: 0.8763 - val_mae: 0.7624
Epoch 5/500
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.2344 - mae: 0.3742 - val_loss: 0.8771 - val_mae: 0.7616
Epoch 6/500
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.1949 - mae: 0.3410 - val_loss: 0.8728 - val_mae: 0.7595
Epoch 7/500
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.1680 - mae: 0.3180 - val_loss: 0.8670 - val_mae: 0.7580
Epoch 8/500
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.1424 - mae: 0.2935 - val_loss: 0.8572 - val_mae: 0.7526
Epoch